In [ ]:
import math


def taylor_maccoll(theta, vr, vtheta, gamma):
    a_sq = 0.5 * (gamma - 1) * (1 - vr**2 - vtheta**2)
    denominator = a_sq - vtheta**2
    if a_sq <= 0 or denominator <= 0:
        raise ValueError("The integration left the regular post-shock flow region.")
    d_vtheta = (
        vr * vtheta**2 - a_sq * (2 * vr + vtheta / math.tan(theta))
    ) / denominator
    return vtheta, d_vtheta


def rk4_step(theta, vr, vtheta, step, gamma):
    k1r, k1t = taylor_maccoll(theta, vr, vtheta, gamma)
    k2r, k2t = taylor_maccoll(
        theta + step / 2, vr + step * k1r / 2, vtheta + step * k1t / 2, gamma
    )
    k3r, k3t = taylor_maccoll(
        theta + step / 2, vr + step * k2r / 2, vtheta + step * k2t / 2, gamma
    )
    k4r, k4t = taylor_maccoll(
        theta + step, vr + step * k3r, vtheta + step * k3t, gamma
    )
    return (
        vr + step * (k1r + 2 * k2r + 2 * k3r + k4r) / 6,
        vtheta + step * (k1t + 2 * k2t + 2 * k3t + k4t) / 6,
    )


def integrate_to_cone(beta, vr, vtheta, gamma, angular_step):
    theta = beta
    minimum_theta = 1e-8
    while theta > minimum_theta:
        step = -min(angular_step, theta - minimum_theta)
        next_vr, next_vtheta = rk4_step(theta, vr, vtheta, step, gamma)
        if next_vtheta >= 0:
            # Locate the zero of vtheta within the last integration step.
            lo, hi = 0.0, 1.0
            for _ in range(36):
                mid = (lo + hi) / 2
                _, mid_vtheta = rk4_step(theta, vr, vtheta, step * mid, gamma)
                if mid_vtheta < 0:
                    lo = mid
                else:
                    hi = mid
            surface_step = step * (lo + hi) / 2
            vr_c, vtheta_c = rk4_step(theta, vr, vtheta, surface_step, gamma)
            return theta + surface_step, math.hypot(vr_c, vtheta_c)
        theta += step
        vr, vtheta = next_vr, next_vtheta
    raise ValueError("No cone-surface crossing was found at a positive polar angle.")


def cone_mach_project(Minf, beta_deg, p_inf, T_inf, gamma=1.4, angular_step=1e-4):
    """Return the post-shock and cone-surface states for a given shock angle."""
    values = (Minf, beta_deg, p_inf, T_inf, gamma, angular_step)
    if not all(math.isfinite(value) for value in values):
        raise ValueError("All inputs must be finite.")
    if Minf <= 1 or gamma <= 1 or min(p_inf, T_inf, angular_step) <= 0:
        raise ValueError("Require M_inf > 1, gamma > 1, and positive pressure, temperature and step size.")

    beta = math.radians(beta_deg)
    mach_angle = math.asin(1 / Minf)
    if not mach_angle < beta < math.pi / 2:
        raise ValueError("The shock angle must be between the Mach angle and 90 degrees.")

    Mn1 = Minf * math.sin(beta)
    normal_mach_sq = Mn1**2
    delta = math.atan(
        (2 / math.tan(beta)) * (normal_mach_sq - 1)
        / (Minf**2 * (gamma + math.cos(2 * beta)) + 2)
    )
    factor = (gamma - 1) / 2
    Mn2 = math.sqrt((1 + factor * normal_mach_sq) / (gamma * normal_mach_sq - factor))
    M2 = Mn2 / math.sin(beta - delta)
    speed2 = (1 + 2 / ((gamma - 1) * M2**2)) ** (-0.5)

    vr2 = speed2 * math.cos(beta - delta)
    vtheta2 = -speed2 * math.sin(beta - delta)
    theta_c, speed_c = integrate_to_cone(beta, vr2, vtheta2, gamma, angular_step)
    Mc = speed_c / math.sqrt(factor * (1 - speed_c**2))

    # Compute the static state immediately behind the shock.
    p2_p1 = 1 + 2 * gamma / (gamma + 1) * (normal_mach_sq - 1)
    rho2_rho1 = (gamma + 1) * normal_mach_sq / ((gamma - 1) * normal_mach_sq + 2)
    p2 = p_inf * p2_p1
    T2 = T_inf * p2_p1 / rho2_rho1

    # The smooth flow between the shock and cone surface is isentropic.
    Tc_T2 = (1 + factor * M2**2) / (1 + factor * Mc**2)
    return {
        "delta_deg": math.degrees(delta),
        "theta_c_deg": math.degrees(theta_c),
        "Mn1": Mn1,
        "Mn2": Mn2,
        "M2": M2,
        "p2": p2,
        "T2": T2,
        "Mc": Mc,
        "pc": p2 * Tc_T2 ** (gamma / (gamma - 1)),
        "Tc": T2 * Tc_T2,
        "rho_c_rho2": Tc_T2 ** (1 / (gamma - 1)),
    }

def read_number(label, default):
    while True:
        entry = input(f"{label} [{default}]: ").strip()
        if not entry:
            return default
        try:
            value = float(entry)
        except ValueError:
            print("Enter a number or press Enter to use the default.")
            continue
        if math.isfinite(value):
            return value
        print("Enter a finite number.")


print("Enter the inputs, or press Enter to use the Example 10–1 defaults.")
M_inf = read_number("Freestream Mach number", 2.0)
beta_deg = read_number("Shock angle beta (degrees)", 30.8503856)
p_inf = read_number("Freestream static pressure (kPa)", 7.0)
T_inf = read_number("Freestream static temperature (K)", 217.0)
gamma = read_number("Specific heat ratio gamma", 1.4)

result = cone_mach_project(M_inf, beta_deg, p_inf, T_inf, gamma)
print(f"Shock angle:               {beta_deg:.6f} deg")
print(f"Post-shock flow deflection: {result['delta_deg']:.6f} deg")
print(f"Cone half-angle:            {result['theta_c_deg']:.6f} deg")
print()
print(f"{'Location':<21} {'Mach number':>11} {'Pressure (kPa)':>15} {'Temperature (K)':>17}")
print(f"{'Immediately post-shock':<21} {result['M2']:>11.6f} {result['p2']:>15.6f} {result['T2']:>17.6f}")
print(f"{'Cone surface':<21} {result['Mc']:>11.6f} {result['pc']:>15.6f} {result['Tc']:>17.6f}")

Enter the inputs, or press Enter to use the Example 10–1 defaults.
Freestream Mach number [2.0]: 
Shock angle beta (degrees) [30.8503856]: 
Freestream static pressure (kPa) [7.0]: 
Freestream static temperature (K) [217.0]: 
Specific heat ratio gamma [1.4]: 
Shock angle:               30.850386 deg
Post-shock flow deflection: 1.047308 deg
Cone half-angle:            9.000000 deg

Location              Mach number  Pressure (kPa)   Temperature (K)
Immediately post-shock    1.962190        7.423419        220.673263
Cone surface             1.857559        8.726597        231.109900
